# Module 2: Units and the Memory Budget

In Module 1 you traced one request and saw that decode runs at tens of tokens per second with a hard ceiling. This module explains that ceiling and the budget behind it. A model is files you can size with arithmetic. By the end you can size a model against a GPU before you touch Kubernetes, and say how many concurrent requests fit.

## Learning objectives
- Read a model's real `config.json` and safetensors metadata and count its parameters
- Compute a model's weight footprint in GB from parameters and bytes per weight
- Compare BF16 and FP8 footprints and see why smaller weights leave more room for cache
- Place the weights and the KV cache in one card's memory and name what is fixed and what is left
- Apply the KV-cache memory formula to the served model and read it per token
- Turn the leftover memory into a concurrency limit by sequence length

## Prerequisites
- Finished Module 1
- Your vLLM Deployment serves `Qwen/Qwen3-4B`; reading the live KV pool needs it running
- About 15 minutes

References: [Qwen3 model cards](https://huggingface.co/Qwen) &middot; [safetensors format](https://github.com/huggingface/safetensors) &middot; [vLLM memory and KV cache](https://docs.vllm.ai/en/latest/design/kv_cache.html) &middot; [Grouped-query attention](https://arxiv.org/abs/2305.13245)

## Memory budget design basics

One card has one pool of device memory (VRAM). Three things share it:

- The weights. A fixed cost set the moment you choose the model and its precision. They do not change as traffic rises.
- The KV cache. The running attention state for every in-flight request. It grows with sequence length and with the number of concurrent requests.
- Overhead. Activations, the CUDA context, and engine bookkeeping.

The weights are paid first. The KV cache is what is left, and it sets how many requests fit at once. That is the whole budget.

![One card split into fixed weights, the KV cache pool, and overhead; FP8 weights shrink and leave more room for KV](images/02_memory_budget_architecture.png)

## 1. Setup

Install the small client deps and make `common/` importable. The inspection cells read the model's real `config.json` and safetensors metadata from the bundled copies under `assets/`, so the numbers are real with no Hub access needed.

In [ ]:
%pip install -q "openai>=1.40" "requests>=2.31"

In [ ]:
# Imports, settings, paths.
import sys
from pathlib import Path

if Path("../common").exists():
    REPO_ROOT = Path("..")
else:
    REPO_ROOT = Path(".")
sys.path.insert(0, str(REPO_ROOT.resolve()))

from common.config import get_settings, print_settings
from common import foundation

settings = get_settings()
settings.model_name = foundation.served_model_name(settings)  # use the model the server actually serves
print_settings(settings)
ASSETS = REPO_ROOT / "02_units_and_memory_budget" / "assets"

SMALL = "Qwen/Qwen3-0.6B"               # a small model, for inspection
BF16 = "Qwen/Qwen3-4B"                  # the model your foundation deployment serves (BF16)
FP8 = "RedHatAI/Qwen3-4B-FP8-dynamic"   # the FP8 build, sized for comparison (Module 5 serves it)
SERVED = settings.model_name            # what your server actually serves, resolved above
print("\ninspecting:", SMALL, "and", BF16)
print("your server currently serves:", SERVED)

**What you should see:** your settings and the model ids. The foundation modules size the 4B you serve, `Qwen/Qwen3-4B`. `SERVED` is read from the environment so the live cells point at whatever your server reports; in this half of the workshop that is the BF16 4B.

## 2. A model is a list of files

A model is not a program. It is a directory: a `config.json` that describes the architecture, one or more safetensors files that hold the numbers, and a tokenizer. Read the config of the small model first. The fields you care about are the layer count, the hidden size, the attention head counts, and the precision.

In [ ]:
# Read the small model's real config from the bundled copy under assets/.
cfg_small = foundation.load_model_config(SMALL, fallback_dir=str(ASSETS))

for k in ["num_hidden_layers", "hidden_size", "num_attention_heads",
          "num_key_value_heads", "head_dim", "vocab_size", "torch_dtype"]:
    print(f"{k:>22}: {cfg_small[k]}")

q, kv = cfg_small["num_attention_heads"], cfg_small["num_key_value_heads"]
print(f"\nGQA: {q} query heads share {kv} key/value heads -> {q // kv}:1")

**What you should see:** the small model's architecture. Read the last two head counts. The query heads outnumber the key/value heads. That ratio is grouped-query attention, and it is the reason the KV cache is small enough to fit. You use it again in section 6.

## 3. Bytes per weight: where the file size comes from

The file size is not magic. It is the parameter count times the bytes each parameter takes. Read the parameter count from the safetensors metadata, the header only, no weights, then multiply by the bytes per weight. BF16 is two bytes.

In [ ]:
# Parameter counts from safetensors metadata (bundled under assets/), by dtype. No weights are read.
def total_params(repo):
    counts = foundation.model_param_count(repo, fallback_dir=str(ASSETS))
    return sum(counts.values()), counts

p_small, _ = total_params(SMALL)
p_bf16, _ = total_params(BF16)

BYTES_BF16 = 2
small_gb = p_small * BYTES_BF16 / 1e9
bf16_gb = p_bf16 * BYTES_BF16 / 1e9

print(f"{SMALL}: {p_small:,} params -> {small_gb:.2f} GB at BF16")
print(f"{BF16}: {p_bf16:,} params -> {bf16_gb:.2f} GB at BF16")
print(f"\nfootprint = parameters x {BYTES_BF16} bytes. That is the fixed weight cost.")

**What you should see:** the small model near 1.5 GB and the served 4B near 8 GB at BF16. Those are the fixed weight costs. The served model's 8 GB is the first claim on the card, before a single request arrives.

## 4. Precision: BF16 against FP8

Precision is bytes per weight. Halve the bytes and you halve the weight footprint. The platform pre-caches an FP8 build of the same 4B model. Read its parameter metadata and compare the footprint. Module 5 serves this FP8 model and measures the result; here you only size it.

In [ ]:
# The FP8 build stores most weights at one byte. Sum the real per-dtype byte cost.
DTYPE_BYTES = {"BF16": 2, "F16": 2, "F32": 4, "F8_E4M3": 1, "F8_E5M2": 1, "I8": 1}

fp8_counts = foundation.model_param_count(FP8, fallback_dir=str(ASSETS))
fp8_bytes = sum(n * DTYPE_BYTES.get(dt, 2) for dt, n in fp8_counts.items())
fp8_gb = fp8_bytes / 1e9

print("FP8 build parameter breakdown by dtype:")
for dt, n in fp8_counts.items():
    print(f"  {dt:>8}: {n:,} params x {DTYPE_BYTES.get(dt, 2)} byte")
print(f"\n{BF16} at BF16 : {bf16_gb:.2f} GB")
print(f"{FP8} : {fp8_gb:.2f} GB")
print(f"saved by FP8       : {bf16_gb - fp8_gb:.2f} GB ({100 * (bf16_gb - fp8_gb) / bf16_gb:.0f}% smaller)")

**What you should see:** the FP8 build noticeably smaller than the BF16 build. Some tensors stay at higher precision, so it is not exactly half; compare the GB, not the param counts. Every GB you save on weights is a GB the KV cache can use. The cost of low precision is quality risk, which is why Module 5 measures before keeping it.

## 5. The GPU and where the weights live

The weights sit in the card's device memory (VRAM). This workshop runs on an RTX 4000 Ada: 20 GB of GDDR6 at about 360 GB/s. That bandwidth sets the decode ceiling you saw in Module 1. To make one token the GPU reads the whole weight set once. The fastest a single stream can decode is then about the bandwidth divided by the weight bytes. The KV read for the sequence is small next to the weights at short context.

In [ ]:
# The workshop card. If you run elsewhere, set these to your card's real numbers.
CARD_VRAM_GB = 20
CARD_BANDWIDTH_GB_S = 360   # RTX 4000 Ada, GDDR6 bandwidth

def decode_ceiling(weight_gb):
    # tok/s <= bandwidth / weight_bytes (one token reads all the weights once).
    return CARD_BANDWIDTH_GB_S / weight_gb

print(f"card: {CARD_VRAM_GB} GB VRAM at {CARD_BANDWIDTH_GB_S} GB/s\n")
print(f"{BF16} BF16 ({bf16_gb:.1f} GB): single-stream decode <= {decode_ceiling(bf16_gb):.0f} tok/s")
print(f"{FP8} ({fp8_gb:.1f} GB): single-stream decode <= {decode_ceiling(fp8_gb):.0f} tok/s")
print("\nThis is an upper bound. It is why decode is the slow phase and why fewer weight bytes help.")

**What you should see:** a single-stream ceiling around 45 tokens per second for the BF16 4B, and higher for FP8 because it reads fewer bytes per token. Compare that to the decode rate you measured in Module 1. Module 4 puts this on the roofline and shows why batching is how you beat the single-stream limit.

## 6. The KV-cache memory formula

The KV cache holds, for every token in a sequence, the key and value vectors at every layer so the next token can attend back without recomputing them. The memory per token is fixed by the architecture:

```
KV bytes per token = 2 (key and value) x layers x kv_heads x head_dim x precision_bytes
```

Read the served model's config and substitute. vLLM's `kv-cache-dtype` defaults to `auto`, which uses the model's dtype, BF16 here, so two bytes per number.

In [ ]:
# Apply the KV-cache memory formula to the served model, from its real config.
cfg = foundation.load_model_config(SERVED, fallback_dir=str(ASSETS))
layers = cfg["num_hidden_layers"]
kv_heads = cfg["num_key_value_heads"]
head_dim = cfg["head_dim"]
q_heads = cfg["num_attention_heads"]
KV_BYTES = 2  # vLLM kv-cache-dtype=auto uses the model dtype (BF16 here)

kv_per_token = 2 * layers * kv_heads * head_dim * KV_BYTES
print(f"{SERVED}: layers={layers}, kv_heads={kv_heads}, head_dim={head_dim}")
print(f"KV bytes/token = 2 x {layers} x {kv_heads} x {head_dim} x {KV_BYTES} "
      f"= {kv_per_token:,} bytes = {kv_per_token / 1024:.0f} KiB/token")

# What GQA buys: multi-head attention would use q_heads instead of kv_heads.
mha = 2 * layers * q_heads * head_dim * KV_BYTES
print(f"\nWithout GQA (one K/V per query head): {mha:,} bytes/token "
      f"-> GQA makes the cache {mha / kv_per_token:.0f}x smaller")

**What you should see:** about 144 KiB per token for the 4B, and that grouped-query attention makes the cache 4x smaller than full multi-head attention would. That single number, KV bytes per token, is what turns leftover memory into a request count in the next section.

## 7. The memory budget: weights against the KV pool

Now put it together on one card. The weights are paid first. What is left, scaled by `gpu-memory-utilization`, becomes the KV pool. Try to read the real pool from your running server; if it is not reachable, compute the budget from the card and the manifest's `--gpu-memory-utilization=0.7`. Either way, the pool in tokens divided by a request's sequence length is how many requests fit.

In [ ]:
# Requires a live vLLM endpoint. Read the live KV pool; fall back to the computed budget if unreachable.
GPU_MEM_UTIL = 0.7   # matches --gpu-memory-utilization in manifests/vllm.yaml

try:
    info = foundation.kv_cache_info(settings)
    pool_tokens = info["capacity_tokens"]
    source = (f"live server: block_size={info['block_size']}, "
              f"num_gpu_blocks={info['num_gpu_blocks']}, prefix_caching={info['enable_prefix_caching']}")
except Exception as e:
    usable_gb = CARD_VRAM_GB * GPU_MEM_UTIL
    kv_budget_gb = usable_gb - bf16_gb - 0.5   # leave ~0.5 GB for activations/overhead
    pool_tokens = int(kv_budget_gb * 1e9 / kv_per_token)
    source = (f"computed budget ({usable_gb:.1f} GB usable - {bf16_gb:.1f} GB weights "
              f"- 0.5 GB overhead = {kv_budget_gb:.1f} GB for KV); server not reachable ({type(e).__name__})")

print(f"KV pool: {pool_tokens:,} tokens")
print(f"source : {source}\n")
print(f"{'context length':>15} | {'KV per request':>14} | {'requests that fit':>18}")
for seq in [1024, 2048, 4096, 8192]:
    per_req_gb = kv_per_token * seq / 1e9
    fits = pool_tokens // seq
    print(f"{seq:>15} | {per_req_gb:>11.2f} GB | {fits:>18}")

**What you should see:** a pool of tens of thousands of tokens, and a table where shorter requests pack many to a card while an 8192-token request leaves room for only a handful. This is the concurrency limit: when more requests arrive than fit, they queue or get preempted, the saturation behavior in Module 7. The live `num_gpu_blocks` is the ground truth the engine computes at startup. If the server is not reachable you see the computed budget instead, labeled in the source line; it predicts that pool rather than reads it.

## 8. The context tax

A request's KV cost is linear in its length, and an agent's context only grows. Every turn re-sends the system prompt, the tool schemas, and the full history, so the KV cost climbs turn over turn until it hits the `max-model-len` ceiling. Size that tax directly.

In [ ]:
# KV cache cost in GB for a range of context lengths and batch sizes.
print(f"KV cache GB (at {kv_per_token / 1024:.0f} KiB/token)\n")
print(f"{'context':>8} | {'1 req':>8} | {'8 reqs':>8} | {'32 reqs':>8}")
for ctx in [1024, 2048, 4096, 8192]:
    row = [kv_per_token * ctx * b / 1e9 for b in (1, 8, 32)]
    print(f"{ctx:>8} | {row[0]:>7.2f} | {row[1]:>7.2f} | {row[2]:>7.2f}")

print(f"\nThe max-model-len on this deployment is 8192 tokens. An agent that grows its prompt "
      f"each turn\nwalks down this table until it hits that ceiling, which caps the task length.")

**What you should see:** the KV cost rising with both context length and batch size. A single 8192-token request is about 1.2 GB. Thirty-two of them at once is about 39 GB, far past the whole pool, which is why long contexts collapse concurrency and the rest queue. This table is also why prefix caching matters: if the shared prefix is cached, those tokens are not re-prefilled or re-stored each turn. Module 3 builds the cache and shows that effect live.

## Things to know

- **Weights are a fixed cost, the KV cache is the variable one.** You pay the weights once; the cache grows with traffic and length.
- **Precision is bytes per weight.** FP8 halves the weight bytes, which speeds decode and frees memory for cache. The cost is quality risk, measured in Module 5.
- **Grouped-query attention keeps the cache affordable.** Fewer key/value heads means a smaller cache per token.
- **The KV pool sets concurrency.** Pool tokens divided by sequence length is how many requests fit. That is the number Module 7 pushes against.
- **Read the pool from the server when you can.** `num_gpu_blocks` times `block_size` is the real capacity; the budget arithmetic predicts it.

## Try it yourself

**Resize the card.** Set `CARD_VRAM_GB` and `CARD_BANDWIDTH_GB_S` in section 5 to an A100 (80 GB, about 2000 GB/s) and re-run sections 5 and 7. The decode ceiling in section 5 changes immediately. The request count in section 7 only moves on the computed-budget path; against a live server it reads the real pool, which does not change.

**Price a longer context.** In section 7, add a 16384-token row. How many requests fit then? **Stretch:** find the context length where only one request fits.

**Quantize the cache.** The KV cache can be stored at one byte instead of two. Set `KV_BYTES = 1` in section 6 and re-run sections 6 and 7. If your server is live, section 7 reads its fixed BF16 pool, so the fits column will not move; to see the effect, compute the predicted pool with `pool_tokens = (CARD_VRAM_GB * GPU_MEM_UTIL - bf16_gb - 0.5) * 1e9 / kv_per_token`, then divide by sequence length.

## Summary

- You read the real Qwen3 configs and counted parameters from safetensors metadata.
- You computed weight footprints from parameters and bytes per weight, and saw FP8 shrink the weight block.
- You placed the weights and the KV cache in one card and named the fixed cost and the leftover.
- You applied the KV-cache memory formula to the served model and read about 144 KiB per token.
- You turned the leftover memory into a concurrency limit by sequence length.

## Next

**Module 3: Prefill, Decode, and the KV Cache.** You sized the cache. Next you build it by hand: work self-attention in numpy, separate prefill from decode, measure time to first token, watch the live cache fill while a request runs, and see a cached prefix come back faster.